In [1]:
import pandas as pd
import glob
import logging
import os
import numpy as np
from frameaxis.core import two_sided_pvalue, fdr_adjust

In [2]:
logging.basicConfig(level=logging.DEBUG,
                    format='%(asctime)s %(name)-12s %(levelname)-8s %(message)s',
                    datefmt="%Y-%m-%d %H:%M:%S")
logger = logging.getLogger(__name__)

# Significant axes

Two-sided test of each axis against the null from notebook 2, with Benjamini-Hochberg correction across all axes within each aspect x sentiment cell. `p` is the two-sided p-value, `q` the BH-adjusted value, and `p_2021` the p-value under the earlier rule (tail chosen by the sign of the score), kept only to document what changed. The effect-size files keep axes with `q <= ALPHA`, sorted by `|diff_a_b|`.

In [3]:
ALPHA = 0.05

def calc_significance(df, COLUMNS, mode):
    BOOTSTRAP_TEMPLATE = "{}/bootstrap_{}_{}.npy"
    OUT_TEMPLATE = "{}/significant_axes_{}_{}.tsv"
    OUT_TEMPLATE2 = "{}/effect_size_significant_axes_{}_{}.tsv"

    for aspect in df['aspect'].unique():
        for sentiment in ['positive', 'negative']:
            df_actual = df.query('aspect == @aspect and sentiment == @sentiment')
            null = np.load(BOOTSTRAP_TEMPLATE.format(mode, aspect.replace("/", "-"), sentiment)).astype(np.float64)

            actual = df_actual[COLUMNS].mean(axis=0).values
            p = two_sided_pvalue(actual, null)
            q = fdr_adjust(p, method="bh")
            p_2021 = np.where(actual >= 0, (null > actual).sum(axis=0), (null < actual).sum(axis=0)) / float(null.shape[0])

            results = pd.DataFrame({"axis": COLUMNS, "diff_a_b": actual - null.mean(axis=0), "p": p, "q": q, "p_2021": p_2021})
            results.sort_values("p", kind="mergesort").to_csv(
                OUT_TEMPLATE.format(mode, aspect.replace("/", "-"), sentiment), sep="\t", index=False)

            significant = results.query('q <= @ALPHA')
            significant = significant.loc[significant["diff_a_b"].abs().sort_values(ascending=False, kind="mergesort").index]
            significant[["axis", "diff_a_b", "p", "q"]].to_csv(
                OUT_TEMPLATE2.format(mode, aspect.replace("/", "-"), sentiment), sep="\t", index=False)

In [4]:
mode = "average"
df = pd.read_csv("big_table_by_{}.tsv".format(mode), sep="\t").dropna()
COLUMNS = [c for c in df.columns if '(' in c]
len(COLUMNS)

1621

In [5]:
calc_significance(df, COLUMNS, mode)

In [6]:
mode = "second_moment"
df = pd.read_csv("big_table_by_{}_with_corpus_mean.tsv".format(mode), sep="\t").dropna()
COLUMNS = [c for c in df.columns if '(' in c]
len(COLUMNS)

1621

In [7]:
calc_significance(df, COLUMNS, mode)